# Raschka Ch. 2 — Working with Text Data

Build: sliding-window dataset + embedding layer.

**Kernel:** select `ai-study/.venv` (Python 3.11) — do not create a local venv here.

In [1]:
import torch
print(torch.__version__)
print("mps:", torch.backends.mps.is_available())

2.11.0
mps: True


In [70]:
import torch.nn.functional as F
y = torch.tensor([1.0])
x1 = torch.tensor([1.1])
w1 = torch.tensor([2.2])
b = torch.tensor([0.0])
z = x1 * w1 + b
a = torch.sigmoid(z)
loss = F.binary_cross_entropy(a, y)

In [10]:
y,x1,w1,b,z,a,loss

(tensor([1.]),
 tensor([1.1000]),
 tensor([2.2000]),
 tensor([0.]),
 tensor([2.4200]),
 tensor([0.9183]),
 tensor(0.0852))

In [37]:
class NeuralNet(torch.nn.Module):
    def __init__(self, num_inputs, num_outputs):
        super().__init__()
        self.layers = torch.nn.Sequential(
          # 1st hidden layer
          torch.nn.Linear(num_inputs, 30),
          torch.nn.ReLU(),
          # 2nd hidden layer
          torch.nn.Linear(30, 20),
          torch.nn.ReLU(),
          # output layer
          torch.nn.Linear(20, num_outputs)
        )
        
    def forward(self, x):
      logits = self.layers(x)
      return logits

In [38]:
torch.manual_seed(123)
model = NeuralNet(num_inputs=50, num_outputs=3)
model

NeuralNet(
  (layers): Sequential(
    (0): Linear(in_features=50, out_features=30, bias=True)
    (1): ReLU()
    (2): Linear(in_features=30, out_features=20, bias=True)
    (3): ReLU()
    (4): Linear(in_features=20, out_features=3, bias=True)
  )
)

In [39]:
[p.numel() for p in model.parameters() if p.requires_grad]

[1500, 30, 600, 20, 60, 3]

In [44]:
model.layers[0].weight

Parameter containing:
tensor([[-0.0577,  0.0047, -0.0702,  ...,  0.0222,  0.1260,  0.0865],
        [ 0.0502,  0.0307,  0.0333,  ...,  0.0951,  0.1134, -0.0297],
        [ 0.1077, -0.1108,  0.0122,  ...,  0.0108, -0.1049, -0.1063],
        ...,
        [-0.0787,  0.1259,  0.0803,  ...,  0.1218,  0.1303, -0.1351],
        [ 0.1359,  0.0175, -0.0673,  ...,  0.0674,  0.0676,  0.1058],
        [ 0.0790,  0.1343, -0.0293,  ...,  0.0344, -0.0971, -0.0509]],
       requires_grad=True)

In [48]:
torch.manual_seed(123)
X = torch.rand((1, 50))
out = model(X)
print(out)


tensor([[-0.1262,  0.1080, -0.1792]], grad_fn=<AddmmBackward0>)


In [49]:
with torch.no_grad():
  out = model(X)
print(out)

tensor([[-0.1262,  0.1080, -0.1792]])


In [52]:
X_train = torch.tensor([
[-1.2, 3.1],
[-0.9, 2.9],
[-0.5, 2.6],
[2.3, -1.1],
[2.7, -1.5]
])
y_train = torch.tensor([0, 0, 0, 1, 1])
X_test = torch.tensor([
[-0.8, 2.8],
[2.6, -1.6],
])
y_test = torch.tensor([0, 1])
X_test, y_test


(tensor([[-0.8000,  2.8000],
         [ 2.6000, -1.6000]]),
 tensor([0, 1]))

In [68]:
# Training loop

import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

train_loader = DataLoader(
    TensorDataset(X_train, y_train),
    batch_size=2,
    shuffle=True,
)

torch.manual_seed(123)
model = NeuralNet(num_inputs=2, num_outputs=3)
optimizer = torch.optim.SGD(model.parameters(), lr=0.5)
num_epochs = 3
for epoch in range(num_epochs):
  model.train()
  for batch_idx, (features, labels) in enumerate(train_loader):
    # Forward pass
    logits = model(features)
    loss = F.cross_entropy(logits, labels)

    # Backward pass
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

  print(f"Epoch: {epoch+1}/{num_epochs}, Loss: {loss.item():.4f}")

# Test loop

model.eval()



Epoch: 1/3, Loss: 0.1067
Epoch: 2/3, Loss: 0.0041
Epoch: 3/3, Loss: 0.0338


NeuralNet(
  (layers): Sequential(
    (0): Linear(in_features=2, out_features=30, bias=True)
    (1): ReLU()
    (2): Linear(in_features=30, out_features=20, bias=True)
    (3): ReLU()
    (4): Linear(in_features=20, out_features=3, bias=True)
  )
)

In [69]:
input_ids = torch.tensor([2, 3, 5, 1])

vocab_size = 6
output_dim = 3

torch.manual_seed(123)
embedding_layer = torch.nn.Embedding(vocab_size, output_dim)
print(embedding_layer.weight)

Parameter containing:
tensor([[ 0.3374, -0.1778, -0.1690],
        [ 0.9178,  1.5810,  1.3010],
        [ 1.2753, -0.2010, -0.1606],
        [-0.4015,  0.9666, -1.1481],
        [-1.1589,  0.3255, -0.6315],
        [-2.8400, -0.7849, -1.4096]], requires_grad=True)


In [67]:
print(embedding_layer(torch.tensor([3, 4])))


tensor([[-0.4015,  0.9666, -1.1481],
        [-1.1589,  0.3255, -0.6315]], grad_fn=<EmbeddingBackward0>)


In [62]:
torch.tensor([3])


tensor([3])

In [ ]:
# I AM NOT DONE
# Exercise: sliding-window dataset and embedding layer (Raschka ch. 2).
# Fill each TODO. Do not edit the tests.
# Done when: each target is the input shifted by one token.

import re

import torch
from torch.utils.data import DataLoader, Dataset

# creates a mapping of words to integers (encoding) and integers to words (decoding)
class SimpleTokenizerV1:
    """Given. Do not edit."""

    def __init__(self, text):
        words = re.findall(r"\w+", text)
        vocab = []
        seen = set()
        for word in words:
            if word not in seen:
                seen.add(word)
                vocab.append(word)
        self.str_to_id = {word: i for i, word in enumerate(vocab)}
        self.id_to_str = {i: word for word, i in self.str_to_id.items()}

    @property
    def vocab_size(self):
        return len(self.str_to_id)

    def encode(self, text):
        words = re.findall(r"\w+", text)
        return [self.str_to_id[word] for word in words]

    def decode(self, ids):
        return " ".join(self.id_to_str[i] for i in ids)

# converts text to a list of integers using tokenizer
class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []
        # Encode txt. tokenizer.encode returns a list of ints.
        token_ids = tokenizer.encode(txt)
        # Loop with range(0, len(token_ids) - max_length, stride).
        for i in range(0, len(token_ids) - max_length, stride):
            # The input chunk starts at i. Its length is max_length.
            input_chunk = token_ids[i:i+max_length]
            # The target chunk starts at i + 1. Its length is max_length.
            target_chunk = token_ids[i+1:i+max_length+1]
            # Append torch.tensor(chunk) to self.input_ids and self.target_ids.
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        # Return the number of windows.
        return len(self.input_ids)

    def __getitem__(self, idx):
        # Return the pair (input tensor, target tensor) at idx.
        return self.input_ids[idx], self.target_ids[idx]

# creates a dataloader from the dataset (ie. a seq of words)
# a dataloader is an enumerable tuple of (input, target) tensors
def create_dataloader_v1(
    txt, batch_size=4, max_length=256, stride=128, shuffle=True, drop_last=True
):
    # Build a SimpleTokenizerV1 on txt.
    tokenizer = SimpleTokenizerV1(txt)
    # Build a GPTDatasetV1. Pass max_length and stride.
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)
    # Return a DataLoader.
    return DataLoader(dataset, batch_size=batch_size, shuffle=shuffle, drop_last=drop_last)

# inputs is a array of token ids. the function embeds each token into a vector space
# it also uses positional encoding to keep track of the position of the tokens
# thse vectors can be used both for model training and inferencing.
def embed_inputs(inputs, vocab_size, output_dim):
    # inputs: LongTensor of shape (batch, context_length)
    #
    # Make a token embedding. Rows: vocab_size. Columns: output_dim.
    token_embedding = torch.nn.Embedding(vocab_size, output_dim)
    # Look up a vector for each token id in inputs.
    token_vectors = token_embedding(inputs)
    # Make a position embedding. Rows: context_length. Columns: output_dim.
    context_length = inputs.shape[1]
    position_embedding = torch.nn.Embedding(context_length, output_dim)
    # Look up a vector for positions 0, 1, ..., context_length - 1.
    position_vectors = position_embedding(torch.arange(context_length))
    # Add the token vectors and the position vectors.
    return token_vectors + position_vectors

# --- tests: do not edit ---

def _test_sliding_window():
    text = "the cat sat on the mat"
    tok = SimpleTokenizerV1(text)
    dataset = GPTDatasetV1(text, tok, max_length=4, stride=1)

    assert len(dataset) == 2, "stride 1 must yield 2 windows"
    inputs, targets = dataset[0]
    assert inputs.tolist() == [0, 1, 2, 3]
    assert targets.tolist() == [1, 2, 3, 0]
    inputs, targets = dataset[1]
    assert inputs.tolist() == [1, 2, 3, 0]
    assert targets.tolist() == [2, 3, 0, 4]

    for idx in range(len(dataset)):
        inputs, targets = dataset[idx]
        assert inputs[1:].tolist() == targets[:-1].tolist(), (
            "target must be the input shifted by one token"
        )

    wide_stride = GPTDatasetV1(text, tok, max_length=4, stride=2)
    assert len(wide_stride) == 1, "stride 2 must yield 1 window"
    print("ok  sliding window")


def _test_dataloader():
    text = "the cat sat on the mat"
    loader = create_dataloader_v1(
        text,
        batch_size=2,
        max_length=4,
        stride=1,
        shuffle=False,
        drop_last=True,
    )
    batches = list(loader)
    assert len(batches) == 1
    inputs, targets = batches[0]
    assert inputs.shape == (2, 4)
    assert targets.shape == (2, 4)
    assert inputs[0].tolist() == [0, 1, 2, 3]
    assert targets[0].tolist() == [1, 2, 3, 0]
    assert torch.equal(inputs[:, 1:], targets[:, :-1])
    print("ok  dataloader")


def _test_embedding():
    torch.manual_seed(0)
    inputs = torch.tensor([[1, 1, 1, 1]])
    vectors = embed_inputs(inputs, vocab_size=5, output_dim=3)
    assert vectors.shape == (1, 4, 3)
    assert vectors.requires_grad
    # The same token id sits in every position.
    # Position vectors must make those rows differ.
    assert not torch.equal(vectors[0, 0], vectors[0, 1])
    print("ok  embedding")


_test_sliding_window()
_test_dataloader()
_test_embedding()
print("ALL TESTS PASSED")


ok  sliding window
ok  dataloader
ok  embedding
ALL TESTS PASSED
